# Decoder Pipeline - 3D U-Net vs V-Net  (HPC full-training twin)

Full-resolution (`256^3`) training of the **3D U-Net** and **V-Net** decoders on the Sunway HPC GPU,
on the **4-channel per-bone** target (femur/tibia/patella/fibula). This is the twin of
`notebooks/modeling/03_decoder_pipeline.ipynb`; the **only** differences are in the CONFIG cell (CUDA
device, 256^3 target, full epochs, mixed precision, gradient checkpointing, data workers). Verify the
pipeline locally first, then run this for the real comparison.

Pipeline: `AP+LAT DRRs -> shared encoder/fusion -> 3D features [64,128,256,512] -> U-Net/V-Net decoder -> per-bone occupancy volumes`

### How to run

0. **First** run `02_frontend_pretrain.ipynb` for the **same `FOLD`** so `models/front_end_fold{FOLD}.pth`
   and `models/decoders/decoder_split_fold{FOLD}.csv` exist. With `REGIME="frozen"` this notebook
   loads and **freezes** that whole front-end, so only the decoder trains. (If the file is missing it
   falls back to a frozen SimCLR encoder + trainable fusion/lift and prints a warning.)
1. Run the cells top to bottom. The **CONFIG** cell is the only place you change settings.
2. **The comparison is a grid.** For each `FOLD` in `0..N_FOLDS-1`, and each `REGIME` in
   `{"frozen", "finetuned"}`, run the whole notebook with `MODEL="unet"`, then `MODEL="vnet"`. Each
   run writes its own `models/decoders/fold{FOLD}/{REGIME}/{MODEL}/` folder and a per-knee metric CSV.
3. When the grid is done, open `04_decoder_comparison.ipynb` to pool the per-knee CSVs and run the
   paired U-Net vs V-Net Wilcoxon test (overall and within the fractured subgroup).

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # reduce CUDA fragmentation (OOM safeguard); must be set before torch initialises CUDA
import math, random, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as cp
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
import nibabel as nib
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# ============================= CONFIG (HPC - full 256^3 training) =============================
# Mirrors the local notebook; only these knobs differ. Run on the Sunway HPC GPU.
ENV          = "HPC"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL        = "unet"        # train "unet", then re-run with "vnet" for the comparison
TARGET_RES   = 256
LIFT_DEPTH   = 16
EPOCHS       = 40
BATCH_SIZE   = 1             # 256^3 is memory-heavy; raise to 2 only if the GPU allows
LR           = 1e-4
CKPT_EVERY   = 5
USE_AMP      = True
USE_GRAD_CKPT = True
NUM_WORKERS  = 4
INCLUDE_GEOMETRIC = False
DEEP_SUPERVISION  = False
# --- per-bone multi-label target (femur/tibia/patella/fibula), one channel per bone ---
N_CLASSES    = 4
BONES        = ["femur", "tibia", "patella", "fibula"]
# --- cross-validation (knee-level, dataset-stratified) ---
N_FOLDS      = 5             # k-fold CV over knees (matches FracReconNet); every knee is tested once
FOLD         = 0             # which fold is held out as TEST this run (0..N_FOLDS-1)
# --- front-end regime (run BOTH and compare in 04_decoder_comparison.ipynb) ---
REGIME       = "frozen"      # "frozen": strict decoder isolation (front-end frozen -> byte-identical
                             #   features for U-Net & V-Net). "finetuned": train encoder+fusion+lift
                             #   +decoder jointly (realistic capacity; NOT a pure decoder ablation).
FREEZE_FRONTEND = (REGIME == "frozen")   # derived from REGIME so the two regimes stay consistent
FREEZE_ENCODER  = (REGIME == "frozen")   # finetuned regime trains the SimCLR backbone too
PRETRAINED   = True
SMOKE_TEST   = False
SMOKE_CASES_PER_GROUP = 3
RESUME_FROM  = None
EXPLICIT_ROOT = None         # e.g. "/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2_24020059"
if DEVICE.type != "cuda":
    print("[warning] CUDA not available - this HPC notebook expects a GPU.")
print("ENV", ENV, "| MODEL", MODEL, "| fold", FOLD, "/", N_FOLDS, "| regime", REGIME,
      "| TARGET_RES", TARGET_RES, "| device", DEVICE, "| epochs", EPOCHS)

In [ ]:
# Resolve the project root robustly (works locally and on HPC, regardless of where
# the notebook is launched from). We look upward for the data/interim/predrr folder,
# which anchors the project (the per-bone GT lives next to it under gt_per_bone_256).
def find_root(start: Path) -> Path:
    if EXPLICIT_ROOT:
        r = Path(EXPLICIT_ROOT)
        if (r / "data" / "interim" / "predrr").exists():
            return r
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "interim" / "predrr").exists():
            return cand
    raise FileNotFoundError("Could not find project root (expected data/interim/predrr). "
                            "Set EXPLICIT_ROOT in the CONFIG cell.")

ROOT           = find_root(Path.cwd())
DATA           = ROOT / "data"
NORMAL_DRR_DIR = DATA / "interim" / "DRRs"                 # AP/LAT DRRs (model inputs)
AUG_DRR_DIR    = DATA / "processed" / "augmented_DRRs"     # augmented DRR variants
PREDRR_DIR     = DATA / "interim" / "predrr"               # bone-windowed CT (root anchor; not the target)
GT_PB_DIR      = DATA / "interim" / "gt_per_bone_256"      # NEW target: per-bone STL-derived GT (femur/tibia/patella/fibula)
MODELS_DIR     = ROOT / "models"
# Encoder fallback (used only if the per-fold front-end from 02 is absent): prefer label-free FCMAE.
FCMAE_CKPT     = MODELS_DIR / "convnextv2_fcmae_encoder.pth"       # label-free FCMAE (+cross-view) encoder
SIMCLR_CKPT    = MODELS_DIR / "convnextv2_simclr_encoder.pth"      # legacy fallback only (retired)
ENCODER_CKPT   = FCMAE_CKPT if FCMAE_CKPT.exists() else SIMCLR_CKPT # fold-agnostic; pretraining uses no labels
FRONTEND_CKPT  = MODELS_DIR / ("front_end_fold%d.pth" % FOLD)      # per-fold frozen front-end (frozen regime only)
# per-fold + per-regime + per-model checkpoints, so the 2 models x 2 regimes per fold never collide
CKPT_DIR       = MODELS_DIR / "decoders" / ("fold%d" % FOLD) / REGIME / MODEL
CKPT_DIR.mkdir(parents=True, exist_ok=True)
GT_CACHE_DIR   = DATA / "interim" / ("gt_per_bone_occ_%d" % TARGET_RES)   # cached (4,T,T,T) per-bone GT at TARGET_RES
print("ROOT:", ROOT)
print("checkpoints ->", CKPT_DIR)

## 1. Shared encoder (copied verbatim from `01_encoder_pipeline.ipynb`)

The encoder is the part both decoders share, so the comparison is fair: **only the decoder
changes**. The code below is copied verbatim from `01_encoder_pipeline.ipynb` so this notebook is
self-contained — **do not edit it here** (keeping it byte-identical is what lets `front_end.pth`
load with `missing=0 unexpected=0`). When `FREEZE_FRONTEND=True`, `build_model()` loads the
pretrained front-end and **freezes all of it** (encoder + fusion + lift).

What it does, in plain terms:
1. A **ConvNeXtV2** backbone turns each X-ray (AP and LAT) into 4 feature maps at increasing depth.
2. **Hybrid bi-planar fusion** merges the two views: cheap convolution at fine scales (keeps local
   fracture detail), cross-attention at coarse scales (aligns global knee shape).
3. A **2D->3D lift** stacks each fused map into a small 3D feature volume (depth = `LIFT_DEPTH`).

Output: a list of 4 multi-scale 3D feature tensors with channels `[64, 128, 256, 512]` — this is
the *contract* the decoder consumes.

In [ ]:
# ===== Encoder front-end - VERBATIM from 01_encoder_pipeline.ipynb. DO NOT EDIT. =====
# (PRETRAINED / FREEZE_ENCODER are set in the CONFIG cell so they stay visible knobs.)
BACKBONE     = "convnextv2_tiny"
IMG_SIZE     = 256
OUT_CHANNELS = [64, 128, 256, 512]
FUSION_TYPES = ["local", "local", "attn", "attn"]   # fine -> coarse

def make_backbone(pretrained=True):
    """features_only ConvNeXtV2 returning 4 multi-scale maps. Falls back to random init offline."""
    try:
        return timm.create_model(BACKBONE, pretrained=pretrained, features_only=True)
    except Exception as e:
        print("[warn] pretrained fetch failed (%s); random init." % type(e).__name__)
        return timm.create_model(BACKBONE, pretrained=False, features_only=True)

FEAT_DIMS = [f["num_chs"] for f in make_backbone(pretrained=False).feature_info]   # [96,192,384,768]

def load_drr(path):
    """npy 256x256 float32 [0,1] -> tensor [3,H,W] (1 channel replicated to 3 for ConvNeXtV2)."""
    arr = np.load(path).astype(np.float32)
    t = torch.from_numpy(arr)
    if t.ndim == 2:
        t = t.unsqueeze(0)
    return t.repeat(3, 1, 1) if t.shape[0] == 1 else t

NORMALIZE = T.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
def paired_tf(t):
    return NORMALIZE(t)

class CrossAttention(nn.Module):
    """AP (query) attends to LAT (key/value). Operates on tokens [B, N, C]."""
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim); self.k = nn.Linear(dim, dim); self.v = nn.Linear(dim, dim)
        self.scale = dim ** -0.5
    def forward(self, a, b):
        attn = F.softmax(torch.matmul(self.q(a), self.k(b).transpose(-2, -1)) * self.scale, dim=-1)
        return torch.matmul(attn, self.v(b)) + a

class LocalFusion(nn.Module):
    """Cheap high-res fusion: concat views + 3x3 conv, residual on AP."""
    def __init__(self, dim):
        super().__init__()
        self.mix = nn.Conv2d(2 * dim, dim, kernel_size=3, padding=1)
    def forward(self, a, b):
        return self.mix(torch.cat([a, b], dim=1)) + a

# --- bi-planar lift orientation (resolved empirically; see Check 1 / _axis_probe) ---
# GT array axes, from nibabel axcodes ('L','P','S'): axis0 = L-R, axis1 = A-P, axis2 = S-I.
# AP projects along A-P (axis1); LAT projects along L-R (axis0). Both DRR rows (H) = S-I (axis2);
# AP cols (W) = L-R (axis0); LAT cols (W) = A-P (axis1). The fused cube is ordered (axis0, axis1,
# axis2) to match the GT array. flip_* reverse a row/col direction vs its volume axis; locked from
# the affine + 1-D S-I profile test and re-confirmed by the one-sample overfit guard.
LIFT_FLIP_SI      = False   # DRR rows  vs axis2 (S-I)
LIFT_FLIP_AP_COL  = False   # AP  cols  vs axis0 (L-R)
LIFT_FLIP_LAT_COL = True    # LAT cols  vs axis1 (A-P)

class BiPlanarFeatureFusion(nn.Module):
    def __init__(self, feat_dims=FEAT_DIMS, out_channels=OUT_CHANNELS,
                 fusion_types=FUSION_TYPES, depth=16, pretrained=True, freeze_encoder=False):
        super().__init__()
        self.encoder = make_backbone(pretrained)
        self.fusion_types = list(fusion_types)
        self.depth = depth   # retained for signature compat; the orthogonal lift no longer uses it
        self.fuse = nn.ModuleList([CrossAttention(d) if t == "attn" else LocalFusion(d)
                                   for d, t in zip(feat_dims, fusion_types)])
        self.to3d = nn.ModuleList([nn.Conv2d(c, o, 1) for c, o in zip(feat_dims, out_channels)])
        # expand3d fuses the two orthogonally back-projected view cubes (2*o -> o) in 3D
        self.expand3d = nn.ModuleList([nn.Conv3d(2 * o, o, 3, padding=1) for o in out_channels])
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
    def load_simclr_encoder(self, path):
        missing, unexpected = self.encoder.load_state_dict(torch.load(path, map_location="cpu"), strict=False)
        print("loaded SimCLR encoder: missing=%d unexpected=%d" % (len(missing), len(unexpected)))
    def _ortho_lift(self, ap_f, lat_f, c2d, c3d):
        """Orthogonal back-projection lift. Each view is placed on the two volume axes it resolves
        and broadcast along its (unobserved) projection axis; the two view cubes are then fused in
        3D. Preserves bi-planar depth instead of extruding one fused 2D map along Z. Output cube is
        ordered (axis0=L-R, axis1=A-P, axis2=S-I) to match the GT array."""
        B, C, H, W = ap_f.shape           # square feature map at this level: S = H = W
        S = H
        ap = c2d(ap_f); lat = c2d(lat_f)  # shared 1x1 projection -> [B, O, H, W] each
        O = ap.shape[1]
        if LIFT_FLIP_SI:      ap = ap.flip(2); lat = lat.flip(2)   # rows (H) = S-I (axis2)
        if LIFT_FLIP_AP_COL:  ap = ap.flip(3)                       # AP  cols (W) = L-R (axis0)
        if LIFT_FLIP_LAT_COL: lat = lat.flip(3)                     # LAT cols (W) = A-P (axis1)
        # AP: (H=axis2, W=axis0) -> cube (axis0, axis1, axis2), broadcast over axis1 (A-P)
        ap_cube = ap.permute(0, 1, 3, 2).unsqueeze(3).expand(B, O, S, S, S)
        # LAT: (H=axis2, W=axis1) -> cube (axis0, axis1, axis2), broadcast over axis0 (L-R)
        lat_cube = lat.permute(0, 1, 3, 2).unsqueeze(2).expand(B, O, S, S, S)
        return c3d(torch.cat([ap_cube, lat_cube], dim=1))           # [B, O, S, S, S]
    def forward(self, ap_img, lat_img):
        ap_feats, lat_feats = self.encoder(ap_img), self.encoder(lat_img)
        fused2d, fused3d = [], []
        for ap_f, lat_f, fuse, c2d, c3d, t in zip(
                ap_feats, lat_feats, self.fuse, self.to3d, self.expand3d, self.fusion_types):
            B, C, H, W = ap_f.shape
            if t == "attn":
                a = ap_f.flatten(2).transpose(1, 2); b = lat_f.flatten(2).transpose(1, 2)
                f2d = fuse(a, b).transpose(1, 2).reshape(B, C, H, W)
            else:
                f2d = fuse(ap_f, lat_f)
            fused2d.append(f2d)                                  # kept only for feature-viz cells
            fused3d.append(self._ortho_lift(ap_f, lat_f, c2d, c3d))
        return fused2d, fused3d

print("encoder feature dims:", FEAT_DIMS)

## 2. Data - paired (DRR inputs, per-bone multi-label GT)

The (X-ray, CT) pairs are aligned *by construction*: the DRRs were rendered from the same knee CT
volumes whose bones were later segmented into per-bone STL meshes and voxelized (see
`00_gt_per_bone.ipynb`). For each DRR pair we look up the matching **per-bone ground truth**.

**Ground-truth target = 4-channel per-bone occupancy** (femur / tibia / patella / fibula), one
binary `{0,1}` channel per bone, from `data/interim/gt_per_bone_256/{dataset}/{key}/{key}_{bone}.nii.gz`.
Each channel is resampled to `TARGET_RES^3` with **nearest-neighbour** (so it stays binary) and the
four are stacked to `(4, T, T, T)`. This replaces the old single-channel `predrr > GT_THRESH`
occupancy, which was density-confounded (dim-CT bone under-captured); the STL masks are solid
cortical+marrow labels, so no intensity threshold is involved. The stacked target is cached to disk
so the niftis are read once, not every epoch.

**Splitting** is done at the *knee* level (`dataset, case, side`) so all augmented variants of one
knee land in the same split (no leakage), and healthy/fractured cases are stratified. We restrict to
knees that actually have per-bone GT on disk (both cohorts), and exclude **geometric** augmentations
(rotations/flips) by default; photometric variants reuse the base GT.

In [ ]:
def build_paired_index():
    """One row per (case, side, variant) with absolute AP/LAT paths + metadata."""
    rows = []
    nmeta = pd.read_csv(NORMAL_DRR_DIR / "drr_generation_metadata.csv")
    for (ds, case, side), _ in nmeta.groupby(["dataset", "case", "side"]):
        ap = NORMAL_DRR_DIR / ds / case / side / "ap.npy"
        lat = NORMAL_DRR_DIR / ds / case / side / "lat.npy"
        if ap.exists() and lat.exists():
            rows.append(dict(dataset=ds, case=case, side=side, variant="normal",
                             geometric=False, ap=str(ap), lat=str(lat)))
    ameta_path = AUG_DRR_DIR / "augmentation_variants_metadata.csv"
    if ameta_path.exists():
        ameta = pd.read_csv(ameta_path)
        for r in ameta.itertuples(index=False):
            ap = AUG_DRR_DIR / r.ap_npy; lat = AUG_DRR_DIR / r.lat_npy
            if ap.exists() and lat.exists():
                rows.append(dict(dataset=r.dataset, case=r.case, side=r.side, variant=r.variant,
                                 geometric=bool(r.geometric), ap=str(ap), lat=str(lat)))
    return pd.DataFrame(rows)

# ---- per-bone GT key: fractured folders carry a "Part" token, VSD healthy do not ----
def key_from(dataset, case, side):
    Side = "Right" if str(side).lower().startswith("r") else "Left"
    return ("%s_Part%s" % (case, Side)) if dataset == "fractured" else ("%s_%s" % (case, Side))

def gt_pb_dir(dataset, case, side):
    return GT_PB_DIR / dataset / key_from(dataset, case, side)

def has_per_bone_gt(dataset, case, side):
    d = gt_pb_dir(dataset, case, side)
    return all((d / ("%s_%s.nii.gz" % (d.name, b))).exists() for b in BONES)

# ---- ground truth: 4-channel per-bone occupancy at TARGET_RES (cached as one .npy per knee) ----
def load_gt_per_bone(dataset, case, side):
    GT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache = GT_CACHE_DIR / ("%s_%s_%s.npy" % (dataset, case, side))
    if cache.exists():
        arr = np.load(cache)                                  # (4, T, T, T) float32
    else:
        d = gt_pb_dir(dataset, case, side); chans = []
        for b in BONES:
            vol = nib.load(str(d / ("%s_%s.nii.gz" % (d.name, b)))).get_fdata().astype(np.float32)
            occ = (vol > 0.5).astype(np.float32)              # STL masks are already binary
            t = F.interpolate(torch.from_numpy(occ)[None, None], size=(TARGET_RES,) * 3, mode="nearest")
            chans.append(t[0, 0].numpy())
        arr = np.stack(chans).astype(np.float32)              # (4, T, T, T)
        np.save(cache, arr)
    return torch.from_numpy(arr)                              # (4, T, T, T)

class PairedDRRVolumeDataset(Dataset):
    """Returns AP/LAT DRRs (3x256x256) + per-bone GT occupancy (4,T,T,T) + metadata."""
    def __init__(self, df, transform=paired_tf):
        self.df = df.reset_index(drop=True); self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        ap = self.transform(load_drr(r.ap)); lat = self.transform(load_drr(r.lat))
        gt = load_gt_per_bone(r.dataset, r.case, r.side)      # (4, T, T, T)
        return {"ap": ap, "lat": lat, "gt": gt,
                "dataset": r.dataset, "case": r.case, "side": r.side, "variant": r.variant}

paired_index = build_paired_index()
if not INCLUDE_GEOMETRIC:
    paired_index = paired_index[~paired_index.geometric].reset_index(drop=True)
# restrict to knees that have per-bone GT built on disk (both cohorts)
paired_index = paired_index[paired_index.apply(
    lambda r: has_per_bone_gt(r.dataset, r.case, r.side), axis=1)].reset_index(drop=True)
print("paired rows (with per-bone GT):", len(paired_index),
      "| knees:", paired_index.groupby(["dataset", "case", "side"]).ngroups,
      "| cases:", paired_index.groupby("dataset")["case"].nunique().to_dict())

# ---- knee-level K-FOLD cross-validation split (dataset-stratified) ----
# Every knee is a test knee in exactly one fold; the split is over (case, side) so all variants of a
# knee share a fold (no leakage). 02_frontend_pretrain.ipynb writes the per-fold CSV so
# encoder/frontend/unet/vnet all use identical fold boundaries for a given FOLD.
def kfold_split(df, n_folds=N_FOLDS, fold=FOLD, seed=SEED):
    rng = random.Random(seed); assign = {}
    for ds, g in df.groupby("dataset"):
        keys = sorted({(r.case, r.side) for r in g.itertuples()})
        rng.shuffle(keys)
        folds = [keys[i::n_folds] for i in range(n_folds)]      # round-robin -> balanced sizes
        test_keys = set(folds[fold % n_folds]); val_keys = set(folds[(fold + 1) % n_folds])
        for k in keys:
            assign[(ds,) + k] = "test" if k in test_keys else ("val" if k in val_keys else "train")
    return df.apply(lambda r: assign[(r.dataset, r.case, r.side)], axis=1)

if SMOKE_TEST:
    # tiny, fast subset just to verify the pipeline runs end-to-end (keep FOLD=0 in smoke)
    keep = paired_index[paired_index.variant == "normal"]
    sub = [g[g.case.isin(list(dict.fromkeys(g.case))[:SMOKE_CASES_PER_GROUP])]
           for _, g in keep.groupby("dataset")]
    paired_index = pd.concat(sub).reset_index(drop=True)

split_csv = MODELS_DIR / "decoders" / ("decoder_split_fold%d.csv" % FOLD)
split_csv.parent.mkdir(parents=True, exist_ok=True)
if split_csv.exists():
    # Load the exact per-fold split written by 02_frontend_pretrain.ipynb to guarantee identical
    # train/val/test assignment across all notebooks (frontend + unet + vnet) for this FOLD.
    saved = pd.read_csv(split_csv)[["dataset", "case", "side", "split"]].drop_duplicates(
        subset=["dataset", "case", "side"])
    paired_index = paired_index.merge(saved, on=["dataset", "case", "side"], how="left")
    n_missing = paired_index["split"].isna().sum()
    if n_missing:
        print("[warn] %d rows not in split CSV; assigning to train. "
              "Re-run 02_frontend_pretrain.ipynb (same FOLD) on the full dataset first." % n_missing)
        paired_index["split"] = paired_index["split"].fillna("train")
    print("loaded fold %d split from %s" % (FOLD, split_csv.name))
else:
    paired_index["split"] = kfold_split(paired_index)
    paired_index[["dataset", "case", "side", "variant", "split"]].to_csv(split_csv, index=False)
    print("[warn] split CSV not found; recomputed fold %d and saved. "
          "Run 02_frontend_pretrain.ipynb (same FOLD) first for a guaranteed-identical split." % FOLD)
print(paired_index.groupby(["split", "dataset"]).size())

In [ ]:
train_df = paired_index[paired_index.split == "train"]
# Evaluate on the CLEAN DRR only: photometric augmented variants are a TRAIN-time augmentation.
# Keeping them in val/test would duplicate each held-out knee as several near-identical rows,
# making the per-knee metric (and the U-Net vs V-Net Wilcoxon pairing) ambiguous.
val_df   = paired_index[(paired_index.split == "val")  & (paired_index.variant == "normal")]
test_df  = paired_index[(paired_index.split == "test") & (paired_index.variant == "normal")]

def make_loader(df, shuffle):
    if len(df) == 0:
        return None
    return DataLoader(PairedDRRVolumeDataset(df), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=NUM_WORKERS, drop_last=False)

train_loader = make_loader(train_df, True)
val_loader   = make_loader(val_df, False)
test_loader  = make_loader(test_df, False)
print("samples -> train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df),
      "(val/test = normal variant only)")

## 3. The decoders — the ONLY difference between the two models

Both decoders use the **same wiring**: they take the encoder's 4 **cube** feature levels
(`8³ → 16³ → 32³ → 64³` from the orthogonal lift), upsample symmetrically (stride-2 on all three
axes) while concatenating the matching encoder feature as a **skip connection** (classic U-Net
shape), then a **super-resolution head** grows the `64³` grid up to the full `TARGET_RES³` output.
Channels taper at high resolution to keep memory affordable.

The single difference is the **building block**:
- **`unet`** -> `DoubleConv`: two `Conv3d -> GroupNorm -> ReLU`. Plain, no residual.
- **`vnet`** -> `VNetResBlock`: two `Conv3d -> GroupNorm -> PReLU` **plus a residual add** (V-Net's
  signature). Residual connections help gradients flow in deep volumetric nets.

Because everything else (encoder, skips, resolution, loss) is identical, any score difference is
attributable to the decoder design.

> **Naming precision (state this in the write-up):** this isolates the *decoder conv-block* (plain
> `DoubleConv` vs. residual `VNetResBlock`) inside a *shared* ConvNeXtV2 encoder–decoder. It is not
> canonical "3D U-Net vs V-Net" — V-Net's learned downsampling and its Dice objective live in the
> shared parts. Report it as a controlled decoder-block ablation, not an architecture comparison.

> **Resolved bottleneck (was the prime confound):** the reference lift fixed the z-axis at
> `LIFT_DEPTH` and grew it to `TARGET_RES` by trilinear interpolation only — it could not synthesise
> depth detail, which capped occupancy Dice ≈ 0.45 *regardless of decoder*. The lift is now an
> **orthogonal back-projection** (AP→axis1, LAT→axis0, fused in 3D) producing genuine `S³` cube
> features, and the decoder upsamples all three axes symmetrically. The shared depth bottleneck is
> gone, so the U-Net/V-Net block difference is no longer masked by it. (`LIFT_DEPTH` is retained
> only for checkpoint/signature compatibility and no longer drives the lift.)

In [ ]:
def conv_block(block_type, in_ch, out_ch):
    return DoubleConv(in_ch, out_ch) if block_type == "unet" else VNetResBlock(in_ch, out_ch)

class DoubleConv(nn.Module):
    """U-Net block: (Conv3d -> GN -> ReLU) x2. Plain, no residual."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)

class VNetResBlock(nn.Module):
    """V-Net block: (Conv3d -> GN -> PReLU) x2 + residual add (input projected if channels differ)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.proj = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.c1 = nn.Conv3d(in_ch, out_ch, 3, padding=1); self.n1 = nn.GroupNorm(8, out_ch); self.a1 = nn.PReLU(out_ch)
        self.c2 = nn.Conv3d(out_ch, out_ch, 3, padding=1); self.n2 = nn.GroupNorm(8, out_ch); self.a2 = nn.PReLU(out_ch)
    def forward(self, x):
        y = self.a1(self.n1(self.c1(x)))
        y = self.n2(self.c2(y))
        return self.a2(y + self.proj(x))

class SuperResHead(nn.Module):
    """Grow the (S, S, S) cube feature grid up to (T,T,T) via staged trilinear upsample + refine
    blocks with tapering channels (heavy work stays at low resolution -> low memory).
    Output: (B, N_CLASSES, T, T, T) - one logit map per bone."""
    def __init__(self, block_type, in_ch, target, use_grad_ckpt=False):
        super().__init__()
        self.target = tuple(int(t) for t in target); self.use_grad_ckpt = use_grad_ckpt
        self.b1 = conv_block(block_type, in_ch, 32)
        self.b2 = conv_block(block_type, 32, 16)
        self.b3 = conv_block(block_type, 16, 8)
        self.out = nn.Conv3d(8, N_CLASSES, 1)   # multi-label: one channel per bone (was 1)
    def _run(self, blk, x):
        if self.use_grad_ckpt and x.requires_grad:
            return cp.checkpoint(blk, x, use_reentrant=False)
        return blk(x)
    def forward(self, x):
        d0, h0, w0 = x.shape[-3:]; dt, ht, wt = self.target
        s1 = (round(d0 + (dt - d0) / 3), round(h0 + (ht - h0) / 3), round(w0 + (wt - w0) / 3))
        s2 = (round(d0 + 2 * (dt - d0) / 3), round(h0 + 2 * (ht - h0) / 3), round(w0 + 2 * (wt - w0) / 3))
        x = F.interpolate(x, size=s1, mode="trilinear", align_corners=False); x = self._run(self.b1, x)
        x = F.interpolate(x, size=s2, mode="trilinear", align_corners=False); x = self._run(self.b2, x)
        x = F.interpolate(x, size=self.target, mode="trilinear", align_corners=False); x = self._run(self.b3, x)
        return self.out(x)

class Decoder3D(nn.Module):
    """Multi-scale skip-connected decoder. Same wiring for both models; only the block differs.
    Inputs are CUBE features from the orthogonal lift (l3=8^3 ... l0=64^3), so the upsamplers are
    symmetric stride-2 on all three axes (8->16->32->64), matching each skip; SuperResHead then
    grows 64^3 -> T^3. Output: (B, N_CLASSES, T, T, T)."""
    def __init__(self, block_type, enc_channels=OUT_CHANNELS, target=(64, 64, 64),
                 use_grad_ckpt=False, deep_supervision=False):
        super().__init__()
        c0, c1, c2, c3 = enc_channels
        self.deep_supervision = deep_supervision; self.target = tuple(int(t) for t in target)
        self.up3 = nn.ConvTranspose3d(c3, c2, kernel_size=2, stride=2)   # 8^3 -> 16^3 (symmetric)
        self.dec3 = conv_block(block_type, c2 + c2, c2)
        self.up2 = nn.ConvTranspose3d(c2, c1, kernel_size=2, stride=2)   # 16^3 -> 32^3
        self.dec2 = conv_block(block_type, c1 + c1, c1)
        self.up1 = nn.ConvTranspose3d(c1, c0, kernel_size=2, stride=2)   # 32^3 -> 64^3
        self.dec1 = conv_block(block_type, c0 + c0, c0)
        self.sr = SuperResHead(block_type, c0, self.target, use_grad_ckpt)
        if deep_supervision:
            # aux heads output N_CLASSES to match the main head (was 1)
            self.aux3 = nn.Conv3d(c2, N_CLASSES, 1); self.aux2 = nn.Conv3d(c1, N_CLASSES, 1); self.aux1 = nn.Conv3d(c0, N_CLASSES, 1)
    def forward(self, feats):
        l0, l1, l2, l3 = feats
        x = self.up3(l3); x = torch.cat([x, l2], 1); x = self.dec3(x); a3 = x
        x = self.up2(x);  x = torch.cat([x, l1], 1); x = self.dec2(x); a2 = x
        x = self.up1(x);  x = torch.cat([x, l0], 1); x = self.dec1(x); a1 = x
        out = self.sr(x)
        if self.deep_supervision and self.training:
            up = lambda h: F.interpolate(h, size=self.target, mode="trilinear", align_corners=False)
            return out, [up(self.aux3(a3)), up(self.aux2(a2)), up(self.aux1(a1))]
        return out, None

class ReconModel(nn.Module):
    """Full model = shared bi-planar encoder/fusion + a (U-Net or V-Net) decoder."""
    def __init__(self, fusion, decoder):
        super().__init__(); self.fusion = fusion; self.decoder = decoder
    def forward(self, ap, lat):
        _, f3d = self.fusion(ap, lat)
        return self.decoder(f3d)

In [ ]:
def build_model():
    # The fusion (encoder + bi-planar fusion + 2D->3D lift) is built once; how we initialise and
    # freeze it depends on the comparison mode.
    fusion = BiPlanarFeatureFusion(depth=LIFT_DEPTH, pretrained=PRETRAINED,
                                   freeze_encoder=(FREEZE_ENCODER and not FREEZE_FRONTEND))
    if FREEZE_FRONTEND and FRONTEND_CKPT.exists():
        # STRICT comparison: load the pretrained front-end and freeze it WHOLESALE, so both the
        # U-Net and V-Net runs consume byte-identical features (only the decoder differs).
        sd = torch.load(FRONTEND_CKPT, map_location="cpu")["front_end"]
        missing, unexpected = fusion.load_state_dict(sd, strict=False)
        for p in fusion.parameters():
            p.requires_grad = False
        print("loaded FROZEN front-end from %s: missing=%d unexpected=%d"
              % (FRONTEND_CKPT.name, len(missing), len(unexpected)))
    elif ENCODER_CKPT.exists():
        # FALLBACK: only the encoder is pretrained (FCMAE preferred); fusion+lift train with the decoder.
        if FREEZE_FRONTEND:
            print("[warn] FREEZE_FRONTEND=True but %s not found; run 02_frontend_pretrain.ipynb first. "
                  "Falling back to encoder-only init (%s)." % (FRONTEND_CKPT.name, ENCODER_CKPT.name))
        fusion.load_simclr_encoder(ENCODER_CKPT)   # FCMAE preferred; loads whatever is at ENCODER_CKPT
    else:
        print("[warn] no front-end or encoder checkpoint; encoder uses ImageNet/random init.")
    decoder = Decoder3D(MODEL, target=(TARGET_RES,) * 3,
                        use_grad_ckpt=USE_GRAD_CKPT, deep_supervision=DEEP_SUPERVISION)
    if USE_GRAD_CKPT and not FREEZE_FRONTEND:
        # FINETUNED regime trains the backbone too, so its two 256^2 forwards (AP, LAT) store
        # activations on top of the 256^3 decoder. Checkpoint the backbone to keep that in memory.
        # (In the frozen regime the encoder needs no activations, so this is skipped.)
        try:
            fusion.encoder.set_grad_checkpointing(True)
            print("encoder gradient checkpointing: ON (finetuned regime)")
        except Exception as e:
            print("[warn] encoder grad checkpointing unavailable (%s)." % type(e).__name__)
    return ReconModel(fusion, decoder)

# shape sanity check (eval mode, no grad -> cheap)
_m = build_model().to(DEVICE).eval()
with torch.no_grad():
    _ap = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    _out, _ = _m(_ap, _ap)
n_train = sum(p.numel() for p in _m.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in _m.parameters() if not p.requires_grad)
print("model:", MODEL, "| output volume:", tuple(_out.shape),
      "| expected:", (1, N_CLASSES, TARGET_RES, TARGET_RES, TARGET_RES))
print("trainable params: %.2fM (decoder only when FREEZE_FRONTEND) | frozen: %.2fM"
      % (n_train / 1e6, n_frozen / 1e6))
del _m, _ap, _out

In [ ]:
# --- NaN diagnostic (optional): finds the first non-finite tensor. Set False to skip. ---
# If training ever prints NaN, run this: it shows whether the inputs, the fused 3D features, or the
# logits become non-finite, and whether float16 autocast (vs fp32) is what introduces it.
RUN_NAN_DIAGNOSTIC = True
if RUN_NAN_DIAGNOSTIC and train_loader is not None:
    _dm = build_model().to(DEVICE).eval()
    _b = next(iter(train_loader))
    _ap, _lat, _gt = _b["ap"].to(DEVICE), _b["lat"].to(DEVICE), _b["gt"].to(DEVICE)
    def _chk(name, t):
        print("  %-12s finite=%s min=%.3g max=%.3g"
              % (name, bool(torch.isfinite(t).all()), float(t.min()), float(t.max())))
    print("inputs:"); _chk("ap", _ap); _chk("lat", _lat); _chk("gt", _gt)
    for use in ([False, True] if DEVICE.type == "cuda" else [False]):
        ad = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        with torch.no_grad():
            if use:
                with torch.amp.autocast("cuda", dtype=ad):
                    _f2, _f3 = _dm.fusion(_ap, _lat); _o, _ = _dm.decoder(_f3)
            else:
                _f2, _f3 = _dm.fusion(_ap, _lat); _o, _ = _dm.decoder(_f3)
        print("autocast", use, "| dtype", (str(ad) if use else "fp32"))
        for i, f in enumerate(_f3):
            _chk("feat3d[%d]" % i, f)
        _chk("logits", _o)
    del _dm, _b, _ap, _lat, _gt

## 4. Loss and metrics

- **Loss = 0.5 * BCE + 0.5 * soft-Dice.** Bone is a small fraction of the volume (~3%), so pure BCE
  is dominated by easy background voxels. Adding Dice directly optimises overlap and handles the
  class imbalance. (Lai's reference used Dice only; adding BCE stabilises early training.)
- **Overlap metrics:** **Dice** and **IoU** on the binarised prediction.
- **Surface metrics (mm):** **HD95** (95th-percentile symmetric surface distance) and **ASSD**
  (average symmetric surface distance). Dice/IoU "evaluate the global regional similarity and may
  neglect the local boundary/surface" ([surface-supervision, arXiv:2405.01204](https://arxiv.org/html/2405.01204v1)),
  but a fracture *is* a boundary phenomenon — so these distance metrics, which
  [FracReconNet (PMC9829664)](https://pmc.ncbi.nlm.nih.gov/articles/PMC9829664/) reports in mm, are
  the ones that actually move for fractures. We convert voxels→mm with `VOXEL_MM` so numbers are
  resolution-independent and comparable to published work.
- All metrics are reported **per knee**, aggregated **mean±std split by healthy vs fractured**, and
  written to a per-knee CSV. The paired U-Net vs V-Net significance test lives in
  `04_decoder_comparison.ipynb`.

We deliberately implement these by hand (instead of pulling in MONAI) so the formulas are visible
and the notebook runs with the libraries already installed.

In [ ]:
class DiceBCEMC(nn.Module):
    """Multi-channel BCE + soft-Dice, averaged over the N_CLASSES bone channels. Reductions run in
    float32 so a 256^3 sum under AMP float16 cannot overflow (>65504 -> inf -> NaN)."""
    def __init__(self, w_bce=0.5, w_dice=0.5, smooth=1.0):
        super().__init__(); self.w_bce = w_bce; self.w_dice = w_dice; self.smooth = smooth
    def _dice(self, logits, target):
        p = torch.sigmoid(logits.float()); t = target.float()
        p = p.reshape(p.size(0), p.size(1), -1); t = t.reshape(t.size(0), t.size(1), -1)
        inter = (p * t).sum(-1)
        d = (2 * inter + self.smooth) / (p.sum(-1) + t.sum(-1) + self.smooth)
        return 1 - d.mean()
    def forward(self, logits, target):
        logits = logits.float()   # compute the loss in fp32 even when the forward pass ran in fp16
        return self.w_bce * F.binary_cross_entropy_with_logits(logits, target.float()) + self.w_dice * self._dice(logits, target)

DICE_BCE = DiceBCEMC()

def total_loss(output, target, aux_weight=0.3):
    out, aux = output
    loss = DICE_BCE(out, target)
    if aux:
        for a in aux:
            loss = loss + aux_weight * DICE_BCE(a, target)
    return loss

@torch.no_grad()
def per_bone_dice_iou(logits, target, thr=0.5):
    """Returns (dice, iou) arrays of shape (B, N_CLASSES) - hard Dice/IoU per bone channel."""
    p = (torch.sigmoid(logits.float()) > thr).float()
    t = (target > 0.5).float()
    p = p.reshape(p.size(0), p.size(1), -1); t = t.reshape(t.size(0), t.size(1), -1)
    inter = (p * t).sum(-1); psum = p.sum(-1); tsum = t.sum(-1)
    dice = (2 * inter + 1e-6) / (psum + tsum + 1e-6)
    iou = (inter + 1e-6) / (psum + tsum - inter + 1e-6)
    return dice.cpu().numpy(), iou.cpu().numpy()

# ---- surface (boundary) metrics, reported in MILLIMETRES (computed per bone channel) ----
# Volumetric Dice/IoU "evaluate the global regional similarity and may neglect the local
# boundary/surface" (Cross-Scale Attention & Surface Supervision, arXiv:2405.01204); fractures ARE
# a boundary phenomenon, so we add the two distance metrics the fracture literature reports:
#   HD95 - 95th-percentile symmetric surface distance (localized disagreement)
#   ASSD - average symmetric surface distance (FracReconNet, PMC9829664, reports this in mm)
# VOXEL_MM converts voxel distances to mm using the GT field-of-view, so numbers are resolution- and
# device-independent and comparable to published work.
def _surface_dists(pred_bin, gt_bin, spacing):
    """Returns (pred-surface->GT distances, GT-surface->pred distances) in mm, or None if a mask is empty."""
    from scipy.ndimage import binary_erosion, distance_transform_edt
    sp = pred_bin & ~binary_erosion(pred_bin); sg = gt_bin & ~binary_erosion(gt_bin)
    if sp.sum() == 0 or sg.sum() == 0:
        return None
    dg = distance_transform_edt(~sg) * spacing   # distance of every voxel to the GT surface
    dp = distance_transform_edt(~sp) * spacing   # distance of every voxel to the pred surface
    return dg[sp], dp[sg]                         # symmetric pair

def hd95(pred_bin, gt_bin, spacing=1.0):
    d = _surface_dists(pred_bin, gt_bin, spacing)
    return float("nan") if d is None else float(np.percentile(np.concatenate(d), 95))

def assd(pred_bin, gt_bin, spacing=1.0):
    d = _surface_dists(pred_bin, gt_bin, spacing)
    return float("nan") if d is None else float(np.concatenate(d).mean())

# mm per voxel at TARGET_RES: the per-bone GT covers a fixed knee field-of-view at 256^3, so
# resampling to TARGET_RES^3 preserves the extent and rescales the spacing. Read it from a real
# per-bone GT affine; fall back to 0.78125 mm @256 (=> 0.78125 * 256 / TARGET_RES) if unavailable.
def _voxel_mm():
    try:
        r = paired_index.iloc[0]
        d = gt_pb_dir(r.dataset, r.case, r.side)
        img = nib.load(str(d / ("%s_%s.nii.gz" % (d.name, BONES[0]))))
        zooms = np.asarray(img.header.get_zooms()[:3], dtype=float)
        dims = np.asarray(img.shape[:3], dtype=float)
        return float(np.mean(zooms * dims / TARGET_RES))   # extent_mm / TARGET_RES, averaged over axes
    except Exception as e:
        print("[warn] could not read GT spacing (%s); using 0.78125mm@256 fallback." % type(e).__name__)
        return 0.78125 * 256.0 / TARGET_RES
VOXEL_MM = _voxel_mm()
print("voxel spacing for surface metrics: %.4f mm" % VOXEL_MM)

In [ ]:
def run_epoch(model, loader, optimizer, scaler, train):
    model.train(train)
    if FREEZE_FRONTEND:
        model.fusion.eval()   # frozen front-end stays in eval: disables timm drop_path so the
                              # features are deterministic & identical across the U-Net/V-Net runs
    use_amp = USE_AMP and DEVICE.type == "cuda"
    # bfloat16 has float32's dynamic range -> no overflow at 65504 (the float16 NaN cause).
    amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
    use_scaler = use_amp and amp_dtype == torch.float16   # only float16 needs GradScaler
    tot, n, steps, skipped = 0.0, 0, 0, 0
    for batch in loader:
        ap = batch["ap"].to(DEVICE); lat = batch["lat"].to(DEVICE); gt = batch["gt"].to(DEVICE)
        with torch.set_grad_enabled(train):
            if use_amp:
                with torch.amp.autocast("cuda", dtype=amp_dtype):
                    loss = total_loss(model(ap, lat), gt)
            else:
                loss = total_loss(model(ap, lat), gt)
        if not torch.isfinite(loss):                       # never backprop a NaN/inf loss
            skipped += 1; optimizer.zero_grad(set_to_none=True); continue
        if train:
            optimizer.zero_grad(set_to_none=True)
            if use_scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                prev = scaler.get_scale(); scaler.step(optimizer); scaler.update()
                if scaler.get_scale() >= prev: steps += 1   # scaler did not skip the step
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); steps += 1
        tot += loss.item() * ap.size(0); n += ap.size(0)
    if skipped: print("  [warn] skipped %d non-finite batch(es)" % skipped)
    return tot / max(n, 1), steps

@torch.no_grad()
def evaluate(model, loader, surface=False):
    """Per-knee metrics: aggregate (mean over the 4 bones) dice/iou [+ hd95_mm/assd_mm if surface],
    plus per-bone columns dice_<bone>/iou_<bone> [+ hd95_mm_<bone>/assd_mm_<bone>]. Surface metrics
    are skipped during per-epoch validation (slow) and computed only for the final test. Returns the
    per-knee DataFrame, the overall (aggregate) mean, and a per-group (healthy vs fractured) table."""
    model.eval(); rows = []
    for batch in loader:
        out, _ = model(batch["ap"].to(DEVICE), batch["lat"].to(DEVICE))
        d, i = per_bone_dice_iou(out, batch["gt"].to(DEVICE))        # (B, N_CLASSES) each
        if surface:
            prob = torch.sigmoid(out.float()).cpu().numpy(); gtn = batch["gt"].numpy()
        for b in range(d.shape[0]):
            rec = dict(dataset=batch["dataset"][b], case=batch["case"][b], side=batch["side"][b],
                       dice=float(d[b].mean()), iou=float(i[b].mean()))
            for k, bone in enumerate(BONES):
                rec["dice_%s" % bone] = float(d[b, k]); rec["iou_%s" % bone] = float(i[b, k])
            if surface:
                hs, as_ = [], []
                for k, bone in enumerate(BONES):
                    pb = prob[b, k] > 0.5; gb = gtn[b, k] > 0.5
                    hh = hd95(pb, gb, VOXEL_MM); aa = assd(pb, gb, VOXEL_MM)
                    rec["hd95_mm_%s" % bone] = hh; rec["assd_mm_%s" % bone] = aa
                    hs.append(hh); as_.append(aa)
                rec["hd95_mm"] = float(np.nanmean(hs)); rec["assd_mm"] = float(np.nanmean(as_))
            rows.append(rec)
    df = pd.DataFrame(rows)
    agg = [c for c in ["dice", "iou", "hd95_mm", "assd_mm"] if c in df.columns]
    overall = df[agg].mean().to_dict() if len(df) else {c: float("nan") for c in agg}
    by = df.groupby("dataset")[agg].agg(["mean", "std"]) if len(df) else None
    return df, overall, by

def save_ckpt(path, model, optimizer, scheduler, epoch, val_metrics):
    torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict() if scheduler else None, "val_metrics": val_metrics,
                "config": {"MODEL": MODEL, "FOLD": FOLD, "N_FOLDS": N_FOLDS, "REGIME": REGIME,
                           "TARGET_RES": TARGET_RES, "LIFT_DEPTH": LIFT_DEPTH,
                           "N_CLASSES": N_CLASSES, "BONES": BONES,
                           "FROZEN_FRONTEND": FREEZE_FRONTEND}}, path)

## 5. Training with checkpointing

Each epoch we train, validate, score Dice/IoU on the validation set, step the cosine LR schedule,
and **save checkpoints**:
- `MODEL_last.pth` — always the most recent (for resuming),
- `MODEL_epochNNN.pth` — every `CKPT_EVERY` epochs (so you can **revisit any epoch** later),
- `MODEL_best.pth` — whenever validation Dice improves,
- `MODEL_history.csv` — per-epoch losses/metrics for the learning-curve plot.

Each checkpoint stores the epoch, model + optimizer + scheduler state, the validation metrics, and
the run config (model type, resolution, lift depth, GT threshold) — everything needed to resume or
to load the model later in the UI. On GPU we use **AMP** (mixed precision) and optional **gradient
checkpointing** to fit `256^3` in memory.

In [ ]:
model = build_model().to(DEVICE)
# when FREEZE_FRONTEND, the whole front-end is frozen -> optimiser holds only the decoder's params
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS, 1))
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))

start_epoch, best_dice, history = 0, -1.0, []
if RESUME_FROM:
    ck = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    if ck.get("scheduler"):
        scheduler.load_state_dict(ck["scheduler"])
    start_epoch = ck["epoch"] + 1
    print("resumed from %s at epoch %d" % (RESUME_FROM, start_epoch))

t0 = time.time()
for epoch in range(start_epoch, EPOCHS):
    tr, steps = run_epoch(model, train_loader, optimizer, scaler, train=True)
    if val_loader:
        va, _ = run_epoch(model, val_loader, optimizer, scaler, train=False)
        _, overall, _ = evaluate(model, val_loader)
    else:
        va, overall = float("nan"), {"dice": float("nan"), "iou": float("nan")}
    if steps > 0:                 # optimizer.step() ran this epoch -> correct order to step scheduler
        scheduler.step()
    history.append(dict(epoch=epoch, train_loss=tr, val_loss=va, val_dice=overall["dice"],
                        val_iou=overall["iou"], lr=optimizer.param_groups[0]["lr"],
                        secs=round(time.time() - t0, 1)))
    pd.DataFrame(history).to_csv(CKPT_DIR / ("%s_history.csv" % MODEL), index=False)
    save_ckpt(CKPT_DIR / ("%s_last.pth" % MODEL), model, optimizer, scheduler, epoch, overall)
    if epoch % CKPT_EVERY == 0:
        save_ckpt(CKPT_DIR / ("%s_epoch%03d.pth" % (MODEL, epoch)), model, optimizer, scheduler, epoch, overall)
    if overall["dice"] > best_dice:
        best_dice = overall["dice"]
        save_ckpt(CKPT_DIR / ("%s_best.pth" % MODEL), model, optimizer, scheduler, epoch, overall)
    print("epoch %03d | train %.4f | val %.4f | val_dice %.4f | best %.4f"
          % (epoch, tr, va, overall["dice"], best_dice))
print("done. checkpoints in", CKPT_DIR)

## 6. Final evaluation (overall + healthy vs fractured)

We reload the **best** checkpoint and score it on the held-out **test fold** (falls back to the
validation split for the smoke test). This writes the **per-knee metric CSV**
(`<MODEL>_test_metrics.csv`) for this `fold`/`regime`/`model`. The full comparison is assembled by
running every fold for `MODEL="unet"` and `MODEL="vnet"` under both `REGIME="frozen"` and
`REGIME="finetuned"`, then opening **`04_decoder_comparison.ipynb`**, which pools all the per-knee CSVs
and runs the paired **Wilcoxon signed-rank** test (U-Net vs V-Net) per metric, overall and within
the fractured subgroup.

In [ ]:
best_path = CKPT_DIR / ("%s_best.pth" % MODEL)
if best_path.exists():
    model.load_state_dict(torch.load(best_path, map_location=DEVICE)["model"])
    print("loaded", best_path.name)

# Final test scoring: Dice/IoU + HD95/ASSD (mm) per knee, aggregated mean+std by group, and the
# per-knee CSV that 04_decoder_comparison.ipynb reads to run the paired U-Net vs V-Net Wilcoxon test.
# (Falls back to the val split for the smoke test, which may have no test cases.)
eval_loader = test_loader or val_loader
if eval_loader:
    df, overall, by = evaluate(model, eval_loader, surface=True)
    df.insert(0, "model", MODEL); df.insert(1, "fold", FOLD); df.insert(2, "regime", REGIME)
    metrics_csv = CKPT_DIR / ("%s_test_metrics.csv" % MODEL)
    df.to_csv(metrics_csv, index=False)
    print("OVERALL:", {k: round(v, 4) for k, v in overall.items()})
    if by is not None:
        print("\nBY GROUP (healthy vs fractured) - mean/std:\n", by.round(4))
    print("\nwrote per-knee metrics ->", metrics_csv)
else:
    print("no eval data in this (smoke) split.")

## 6b. Reconstruction validation — Check 3 + 3D showcase

After scoring, validate the actual reconstructions:

- **Check 3 — prediction sanity.** Flags any predicted volume that collapsed (near-empty/full) or
  extruded (constant along an axis). Warns on the smoke run (undertrained); must be clean after real
  training.
- **3D showcase.** Marching-cubes **surface renders** (prediction vs GT) plus axial/coronal/sagittal
  **mid-slice overlays** (pred=red, GT=green, overlap=yellow) for the worst / median / best knees by
  Dice (and a fractured case), annotated with Dice/IoU/HD95/ASSD. PNGs saved under
  `CKPT_DIR/showcase_reconstructions/`.

In [ ]:
# ===== Check 3 - prediction sanity (collapse / extrusion guard) =====
# A correct reconstruction is neither near-empty/near-full (collapse) nor constant along an axis
# (extrusion). We report the occupancy fraction and per-axis std of the predicted bone UNION (max
# over the 4 bone channels) and FLAG anything suspect. On a short smoke run an undertrained model can
# predict near-empty, so this WARNS rather than hard-fails here; after real training it must report 0
# flagged (Checks 1-2 are the hard asserts on orientation/features).
if eval_loader:
    flags = []
    model.eval()
    with torch.no_grad():
        for b in eval_loader:
            out, _ = model(b["ap"].to(DEVICE), b["lat"].to(DEVICE))
            u = (torch.sigmoid(out.float()) > 0.5).float().amax(dim=1)   # union over bones -> (B,T,T,T)
            for j in range(u.shape[0]):
                v = u[j]; frac = v.mean().item()
                s0, s1, s2 = v.std(0).mean().item(), v.std(1).mean().item(), v.std(2).mean().item()
                collapse = frac < 1e-4 or frac > 0.99
                extrude = (frac > 1e-4) and (min(s0, s1, s2) < 1e-6)
                flags.append((b["dataset"][j], b["case"][j], b["side"][j], frac, s0, s1, s2, collapse or extrude))
    print("Check 3 - per-prediction bone-union occupancy fraction + per-axis std:")
    for ds, case, side, frac, s0, s1, s2, bad in flags:
        print("  %-9s %-8s %-5s frac=%.4f std=(%.4f,%.4f,%.4f)  %s"
              % (ds, case, side, frac, s0, s1, s2, "FLAG (collapse/extrusion)" if bad else "ok"))
    n_bad = sum(f[7] for f in flags)
    if n_bad:
        print("[warn] %d/%d predictions look collapsed/extruded - expected on a 2-epoch smoke run, "
              "but MUST be 0 after real training." % (n_bad, len(flags)))
    else:
        print("Check 3 PASS: all predictions have a sane occupancy fraction and vary on all axes.")
else:
    print("no eval data (smoke split) - Check 3 skipped.")

In [ ]:
# ===== Post-decode 3D reconstruction showcase (marching-cubes surface + tri-plane) =====
# For a few held-out knees (worst / median / best by Dice, plus a fractured case if present) we
# render the predicted vs GT bone surface and the three mid-slices, annotated with the metrics. The
# 4 per-bone channels are combined into a single bone UNION for the surface + overlay. PNGs are
# written to CKPT_DIR/showcase_reconstructions/ for the write-up.
from skimage.measure import marching_cubes
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

SHOWCASE_DIR = CKPT_DIR / "showcase_reconstructions"; SHOWCASE_DIR.mkdir(parents=True, exist_ok=True)

def _lookup_paths(dataset, case, side):
    sub = paired_index[(paired_index.dataset == dataset) & (paired_index.case == case)
                       & (paired_index.side == side) & (paired_index.variant == "normal")]
    return (sub.iloc[0].ap, sub.iloc[0].lat) if len(sub) else (None, None)

def _surface(ax, vol, color, title):
    """Render a marching-cubes surface. vol may be binary (GT) or float prob [0,1] (pred)."""
    ax.set_title(title, fontsize=9)
    # Use > 0.5 check so this works for both binary (0/1) and float-prob volumes.
    if (vol > 0.5).sum() < 10:
        ax.text2D(0.5, 0.5, "(empty)", ha="center", transform=ax.transAxes); ax.set_axis_off(); return
    v = vol
    if v.shape[0] > 128:   # downsample big volumes so matplotlib 3D stays responsive
        v = F.interpolate(torch.from_numpy(v.astype(np.float32))[None, None], size=(128,) * 3,
                          mode="trilinear", align_corners=False)[0, 0].numpy()
    try:
        # Adaptive level: marching_cubes requires level to be strictly between min and max.
        # A binary all-1 array (positive collapse) has min=max=1 and causes a ValueError at
        # level=0.5. Using float probabilities (passed from the caller) widens the range;
        # the adaptive fallback handles any remaining edge case (near-total collapse).
        level = 0.5
        if v.min() >= level or v.max() <= level:
            level = float(v.min()) + 0.5 * (float(v.max()) - float(v.min()))
        verts, faces, _, _ = marching_cubes(v, level=level)
        mesh = Poly3DCollection(verts[faces], alpha=0.6); mesh.set_facecolor(color)
        ax.add_collection3d(mesh)
        ax.set_xlim(0, v.shape[0]); ax.set_ylim(0, v.shape[1]); ax.set_zlim(0, v.shape[2])
        ax.view_init(elev=15, azim=-70)
        if level != 0.5:
            ax.set_title(title + "\n(level=%.2f, collapsed pred)" % level, fontsize=8)
    except Exception as e:
        ax.text2D(0.5, 0.5, "marching_cubes failed:\n%s" % type(e).__name__, ha="center",
                  transform=ax.transAxes)
    ax.set_axis_off()

def _triplane(axs, gt, pred_bin):
    mid = [s // 2 for s in gt.shape]
    for col, (axis, name) in enumerate(zip((0, 1, 2), ("axis0 L-R", "axis1 A-P", "axis2 S-I"))):
        g = np.take(gt, mid[axis], axis=axis); p = np.take(pred_bin, mid[axis], axis=axis)
        rgb = np.stack([p, g, np.zeros_like(g)], -1)   # pred=red, GT=green, overlap=yellow
        axs[col].imshow(np.clip(rgb, 0, 1)); axs[col].set_title("%s mid" % name, fontsize=8); axs[col].axis("off")

if eval_loader and len(df):
    s = df.sort_values("dice").reset_index(drop=True)
    picks = [("worst", s.iloc[0]), ("median", s.iloc[len(s) // 2]), ("best", s.iloc[-1])]
    frac = df[df.dataset == "fractured"].sort_values("dice")
    if len(frac) and not any(p[1].dataset == "fractured" for p in picks):
        picks.append(("fractured", frac.iloc[-1]))
    model.eval()
    for tag, row in picks:
        ap_p, lat_p = _lookup_paths(row.dataset, row.case, row.side)
        if ap_p is None:
            continue
        with torch.no_grad():
            out, _ = model(paired_tf(load_drr(ap_p)).unsqueeze(0).to(DEVICE),
                           paired_tf(load_drr(lat_p)).unsqueeze(0).to(DEVICE))
        # Combine the 4 bone channels into a single occupancy union. Use float sigmoid probabilities
        # (max over bones) for marching cubes (avoids ValueError when a binary array is all 1s under
        # positive collapse). Keep the binary union for the tri-plane overlay.
        pred_prob = torch.sigmoid(out[0].float()).amax(0).cpu().numpy()   # float [0,1] union
        pred_bin  = (pred_prob > 0.5).astype(np.float32)                  # binary union for tri-plane
        gt = load_gt_per_bone(row.dataset, row.case, row.side).numpy().max(0)   # GT union
        fig = plt.figure(figsize=(15, 4.2))
        _surface(fig.add_subplot(1, 5, 1, projection="3d"), gt.astype(np.float32), "tab:green", "GT surface")
        _surface(fig.add_subplot(1, 5, 2, projection="3d"), pred_prob, "tab:red", "pred surface")
        _triplane([fig.add_subplot(1, 5, 3 + i) for i in range(3)], gt, pred_bin)
        sm = "  ".join("%s=%.3f" % (k, row[k]) for k in ("dice", "iou") if k in row)
        sm2 = "  ".join("%s=%.1fmm" % (k, row[k]) for k in ("hd95_mm", "assd_mm")
                        if k in row and np.isfinite(row[k]))
        fig.suptitle("%s  -  %s %s %s   (%s   %s)   [pred=red  GT=green  overlap=yellow, bone union]"
                     % (tag, row.dataset, row.case, row.side, sm, sm2), fontsize=10)
        plt.tight_layout()
        png = SHOWCASE_DIR / ("%s_%s_%s_%s_%s.png" % (tag, row.dataset, row.case, row.side, MODEL))
        fig.savefig(png, dpi=110, bbox_inches="tight"); plt.show()
        print("saved", png.name)
    print("\nshowcase reconstructions ->", SHOWCASE_DIR)
else:
    print("no eval data (smoke split) - showcase skipped.")

In [ ]:
h = pd.read_csv(CKPT_DIR / ("%s_history.csv" % MODEL))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val")
ax[0].set_title("%s loss" % MODEL); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(h.epoch, h.val_dice, label="val Dice"); ax[1].plot(h.epoch, h.val_iou, label="val IoU")
ax[1].set_title("%s val metrics" % MODEL); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 7. Quality-assurance views

**(A) Per-bone GT preview.** The four stacked bone channels (femur/tibia/patella/fibula) for one
knee, with each channel's occupancy fraction - a quick check that all four bones are present and
sensibly sized (patella/fibula small; femur/tibia large).

**(B) Reconstruction preview.** Mid-slices of the predicted bone *union* next to the GT union, as a
quick visual sanity check (rough after a 2-epoch smoke run - expected).

**(C) Per-bone learnability capacity check (optional).** Absorbed from the retired
`gt_per_bone_modelling.ipynb`: overfit 2 knees decoder-only and confirm every bone channel is
learnable independently. Off by default (set the flag in the cell).

In [ ]:
# (A) per-bone GT preview: the 4 stacked bone channels for one knee (mid-coronal)
sample = paired_index.iloc[0]
gt4 = load_gt_per_bone(sample.dataset, sample.case, sample.side).numpy()   # (4, T, T, T)
mid = gt4.shape[2] // 2
fig, ax = plt.subplots(1, 4, figsize=(14, 4))
for j, bone in enumerate(BONES):
    ax[j].imshow(gt4[j, :, mid, :], cmap="gray", origin="lower")
    ax[j].set_title("%s  (%.2f%%)" % (bone, gt4[j].mean() * 100)); ax[j].axis("off")
plt.suptitle("%s %s %s - per-bone GT (mid-coronal)" % (sample.dataset, sample.case, sample.side))
plt.tight_layout(); plt.show()

# (B) reconstruction preview vs GT (bone union over the 4 channels)
if eval_loader:
    batch = next(iter(eval_loader))
    with torch.no_grad():
        out, _ = model(batch["ap"].to(DEVICE), batch["lat"].to(DEVICE))
    pred = (torch.sigmoid(out[0].float()).amax(0).cpu().numpy() > 0.5).astype(float)
    gtv = batch["gt"][0].numpy().max(0)
    m = pred.shape[0] // 2
    fig, ax = plt.subplots(1, 3, figsize=(11, 4))
    ax[0].imshow(batch["ap"][0, 0], cmap="gray"); ax[0].set_title("input AP")
    ax[1].imshow(gtv[m], cmap="gray"); ax[1].set_title("GT union mid-slice")
    ax[2].imshow(pred[m], cmap="gray"); ax[2].set_title("prediction union mid-slice")
    for a in ax:
        a.axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
# (C) per-bone learnability capacity check - overfit 2 knees, decoder-only (absorbed from
# the retired gt_per_bone_modelling.ipynb). Proves the encoder-decoder can represent each of the 4
# bones as an independent label: freeze the front-end, cache the fused 3D features once, and overfit
# ONLY the decoder (batch 1) until per-bone Dice climbs high. Capacity sanity-check, NOT part of the
# CV comparison - set the flag to run it.
RUN_LEARNABILITY_SMOKE = False
if RUN_LEARNABILITY_SMOKE:
    sdf = (paired_index[paired_index.variant == "normal"]
           .drop_duplicates(["dataset", "case", "side"]).head(2).reset_index(drop=True))
    print("capacity knees:", [key_from(r.dataset, r.case, r.side) for r in sdf.itertuples()])
    cap_model = build_model().to(DEVICE); cap_model.eval()
    cache = []
    with torch.no_grad():
        for r in sdf.itertuples():
            ap = paired_tf(load_drr(r.ap))[None].to(DEVICE)
            lat = paired_tf(load_drr(r.lat))[None].to(DEVICE)
            _, f3d = cap_model.fusion(ap, lat)
            gt = load_gt_per_bone(r.dataset, r.case, r.side)[None].to(DEVICE)
            cache.append(([f.detach() for f in f3d], gt))
    opt = torch.optim.Adam([p for p in cap_model.decoder.parameters() if p.requires_grad], lr=1e-3)
    hist = []
    for ep in range(200):
        cap_model.decoder.train()
        for f3d, gt in cache:
            opt.zero_grad(); out, _ = cap_model.decoder(f3d)
            loss = DICE_BCE(out, gt); loss.backward(); opt.step()
        if ep % 20 == 0 or ep == 199:
            with torch.no_grad():
                ds = np.mean([per_bone_dice_iou(cap_model.decoder(f3d)[0], gt)[0][0]
                              for f3d, gt in cache], 0)
            hist.append((ep, *ds))
            print("ep%3d | " % ep + " ".join("%s %.3f" % (b, ds[j]) for j, b in enumerate(BONES)))
            if (ds >= 0.90).all():
                print("early stop: all bones >= 0.90"); break
    final = np.array(hist[-1][1:])
    print("final per-bone Dice:", dict(zip(BONES, final.round(3))))
    assert (final >= 0.80).all(), "some bone underfit: %s" % dict(zip(BONES, final.round(3)))
    print("CAPACITY PASS: all 4 bones learnable as independent labels.")
else:
    print("learnability capacity check skipped (set RUN_LEARNABILITY_SMOKE=True to run).")

## Next steps

- Train `MODEL = "unet"` to convergence, then re-run with `MODEL = "vnet"`.
- Compare the two `*_history.csv` files and the per-knee `*_test_metrics.csv` (now carrying per-bone
  `dice_<bone>` columns); pool them across folds/regimes in `04_decoder_comparison.ipynb`.
- Copy the `models/decoders/` checkpoints back to your machine and explore them in `05_decoder_ui.ipynb`.
- If `256^3` runs out of GPU memory: keep `BATCH_SIZE = 1`, ensure `USE_AMP` and `USE_GRAD_CKPT`
  are `True`, or temporarily set `FREEZE_ENCODER = True` to cut activation memory.